<a href="https://colab.research.google.com/github/ShayanJs2005/AAI2025/blob/2026fall/Predict_Customer_Churn(New_Code).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data Source:
# Synthetic customer dataset created for this assignment with 150 records.

# Generate larger customer data
np.random.seed(42)

num_customers = 150

data = {
    'age': np.random.randint(18, 70, num_customers),
    'monthly_usage_hours': np.random.randint(5, 80, num_customers),
    'purchase_amount': np.random.randint(50, 500, num_customers),
    'customer_service_calls': np.random.randint(0, 10, num_customers),
    'region': np.random.choice(
        ['North', 'South', 'West', 'East'],
        num_customers
    )
}

df = pd.DataFrame(data)

# Create churn values
df['churn'] = (
    (
        (df['monthly_usage_hours'] < 25) |
        (df['customer_service_calls'] >= 6)
    )
).astype(int)

# Features and target
X = df[
    [
        'age',
        'monthly_usage_hours',
        'purchase_amount',
        'customer_service_calls',
        'region'
    ]
]

y = df['churn']

# Preprocessing: Scale numerical features and one-hot encode region
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            [
                'age',
                'monthly_usage_hours',
                'purchase_amount',
                'customer_service_calls'
            ]
        ),
        (
            'cat',
            OneHotEncoder(sparse_output=False),
            ['region']
        )
    ]
)

# Create pipeline with preprocessing and logistic regression
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42))
    ]
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'age': [35],
    'monthly_usage_hours': [20],
    'purchase_amount': [150],
    'customer_service_calls': [5],
    'region': ['West']
})

churn_probability = model.predict_proba(new_customer)[0][1]

# Classify based on threshold
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(['region'])
).tolist() + [
    'age',
    'monthly_usage_hours',
    'purchase_amount',
    'customer_service_calls'
]

coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Churn Probability for new customer: 0.91
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
region_East: 0.52
region_North: -1.52
region_South: -0.03
region_West: 2.63
age: -0.17
monthly_usage_hours: -0.29
purchase_amount: 0.69
customer_service_calls: -0.23
